# 02 - Bins and WoE

I split the 2007 loans 60/20/20, fit the bins on training only, and carry those bins forward. The candidate inputs are the 2005 origination fields.

DTI 999 gets its own treatment. I keep the bin tables alongside the selected fields so I can see what the IV screen actually did.


In [1]:
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from optbinning import BinningProcess
from sklearn.model_selection import train_test_split
from cohort import NUMERIC as NUM

LOCAL_DATA = Path(r"E:\CRM Projects\Data")
if "POCKET_CRM_DATA" not in os.environ and LOCAL_DATA.is_dir():
    os.environ["POCKET_CRM_DATA"] = str(LOCAL_DATA)
DATA = Path(os.environ.get("POCKET_CRM_DATA", "./data")).expanduser().resolve()
OUT = DATA / "pocket1_model"
TARGET, SEED = "default_180_plus_2007", 42
df = pd.read_parquet(DATA / "model_2005q1_dec2006_2007.parquet")
# Origination fields to try in the scorecard.
CAT = ["occupancy_status", "channel", "property_type", "loan_purpose", "seller_name", "interest_only_indicator"]
FEATURES = NUM + CAT


In [2]:
# Split before fitting bins; optbinning uses np.nan for missing categories.
train, rest = train_test_split(df, test_size=.40, stratify=df[TARGET], random_state=SEED)
valid, test = train_test_split(rest, test_size=.50, stratify=rest[TARGET], random_state=SEED)
splits = {"train":train, "validation":valid, "test":test}

def predictors(loans):
    x = loans[FEATURES].copy()
    x[NUM] = x[NUM].astype("float64")
    x[CAT] = x[CAT].astype(object).where(x[CAT].notna(), np.nan)
    return x

X = {name: predictors(loans) for name, loans in splits.items()}
print(pd.DataFrame({k:d[TARGET].agg(["size", "sum", "mean"]) for k,d in splits.items()}).T)


                size    sum      mean
train       171712.0  338.0  0.001968
validation   57238.0  113.0  0.001974
test         57238.0  112.0  0.001957


In [3]:
# Use broad bins. Defaults are scarce, and DTI 999 is a special code.
params = {f:{"min_bin_n_event":20} for f in FEATURES}
params["dti"]["special_codes"] = [999]
bp = BinningProcess(variable_names=FEATURES, categorical_variables=CAT,
                    max_n_bins=5, min_bin_size=.05, binning_fit_params=params)
bp.fit(X["train"], train[TARGET].astype(int))
summary = bp.summary().set_index("name")
print(summary[["status", "n_bins", "iv"]].round(4))
if not summary["status"].isin(["OPTIMAL", "FEASIBLE"]).all():
    raise RuntimeError("Binning failed for some variables; inspect the summary")


                          status n_bins        iv
name                                             
fico                     OPTIMAL      5  0.887492
cltv                     OPTIMAL      5  0.462881
dti                      OPTIMAL      5  0.166613
original_upb             OPTIMAL      4  0.023336
ltv                      OPTIMAL      5  0.498336
original_interest_rate   OPTIMAL      5  0.399757
original_loan_term       OPTIMAL      3  0.214125
occupancy_status         OPTIMAL      1       0.0
channel                  OPTIMAL      2  0.015544
property_type            OPTIMAL      3  0.014324
loan_purpose             OPTIMAL      3   0.01573
seller_name              OPTIMAL      5  0.133868
interest_only_indicator  OPTIMAL      1       0.0


In [4]:
# Thin missing/special groups get neutral WoE. I keep their counts below.
tables = []
for f in FEATURES:
    t = bp.get_binned_variable(f).binning_table.build(add_totals=False).copy()
    t.insert(0, "feature", f)
    t["bin_type"] = t["Bin"].astype(str).str.lower().map(
        lambda s: "missing" if s == "missing" else ("special" if s == "special" else "regular"))
    t["thin_bin"] = (pd.to_numeric(t["Count"], errors="coerce").gt(0) &
                     ((pd.to_numeric(t["Event"], errors="coerce") < 20) |
                      (pd.to_numeric(t["Non-event"], errors="coerce") < 20)))
    tables.append(t)
bin_report = pd.concat(tables, ignore_index=True)
neutral_bins = sorted({(r["feature"], r["bin_type"]) for _, r in bin_report.loc[
    bin_report["thin_bin"] & bin_report["bin_type"].ne("regular")].iterrows()})
print("Thin populated bins (including missing/special):")
print(bin_report.loc[bin_report["thin_bin"],
                     ["feature", "Bin", "Count", "Event", "Non-event", "bin_type"]].to_string(index=False))
print("Neutral WoE=0 for thin missing/special groups:", neutral_bins)
# Save the fitted WoE and the value used in the model.
bin_report["WoE_used"] = pd.to_numeric(bin_report["WoE"], errors="coerce")
for f, kind in neutral_bins:
    bin_report.loc[bin_report["feature"].eq(f) &
                   bin_report["bin_type"].eq(kind), "WoE_used"] = 0.0


Thin populated bins (including missing/special):
feature     Bin  Count  Event  Non-event bin_type
   fico Missing    136      2        134  missing
   cltv Missing     11      0         11  missing
    dti Special   5321      9       5312  special
    ltv Missing     11      0         11  missing
Neutral WoE=0 for thin missing/special groups: [('cltv', 'missing'), ('dti', 'special'), ('fico', 'missing'), ('ltv', 'missing')]


In [5]:
# Same fitted bins, including the thin-bin adjustment, on every split.
WOE_OPTS = dict(metric="woe", metric_missing="empirical", metric_special="empirical")
woe = {}
for name, x in X.items():
    w = pd.DataFrame(np.asarray(bp.transform(x, **WOE_OPTS)), index=x.index, columns=FEATURES)
    for f, kind in neutral_bins:
        mask = (x[f].isna() if kind == "missing" else
                (x[f].eq(999).fillna(False) if f == "dti" else pd.Series(False, index=x.index)))
        w.loc[mask.to_numpy(dtype=bool), f] = 0.0
    if not np.isfinite(w.to_numpy(dtype=float)).all():
        raise ValueError(f"{name}: nonfinite WoE values")
    woe[name] = w


In [6]:
# Working screen: IV >= .02; for highly correlated fields, keep the higher IV.
iv = summary["iv"].astype(float).sort_values(ascending=False)
candidates = [f for f in iv.index if iv[f] >= .02 and woe["train"][f].nunique() > 1]
corr = woe["train"][candidates].corr().abs()
selected = []
for f in candidates:
    if all(corr.loc[f, prev] < .80 for prev in selected):
        selected.append(f)
if not selected:
    raise RuntimeError("No predictors passed train IV/correlation screening")
print("Selected features and IV:\n", iv.loc[selected].round(4))


Selected features and IV:
 name
fico                      0.8875
ltv                       0.4983
original_interest_rate    0.3998
original_loan_term        0.2141
dti                       0.1666
seller_name               0.1339
original_upb              0.0233
Name: iv, dtype: float64


In [7]:
# Save the exact split and binning setup used by the next notebooks.
OUT.mkdir(exist_ok=True)
for name, loans in splits.items():
    export = loans[["loan_id", TARGET]].copy()
    export[selected] = woe[name][selected].to_numpy()
    export.to_parquet(OUT / f"woe_{name}.parquet", index=False)
    # NB4 uses these labels for PSI.
    bins = pd.DataFrame(np.asarray(bp.transform(X[name], metric="bins")),
                        index=X[name].index, columns=FEATURES).astype("string")
    bins[selected].assign(loan_id=loans["loan_id"].to_numpy()).to_parquet(
        OUT / f"bins_{name}.parquet", index=False)
pd.concat([d[["loan_id"]].assign(split=name) for name,d in splits.items()],
          ignore_index=True).to_parquet(OUT / "loan_splits.parquet", index=False)
summary.to_csv(OUT / "iv_summary.csv")
bin_report.to_csv(OUT / "binning_tables.csv", index=False)
joblib.dump(dict(binning_process=bp, features=FEATURES, selected=selected, target=TARGET,
                 seed=SEED, woe_options=WOE_OPTS, neutral_woe_bins=neutral_bins),
            OUT / "feature_pipeline.joblib")
print("Saved WoE and bin-label splits, binning tables, IV report and pipeline to", OUT)


Saved WoE and bin-label splits, binning tables, IV report and pipeline to E:\CRM Projects\Data\pocket1_model
